In [148]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.utils.class_weight import compute_class_weight
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.decomposition import PCA,TruncatedSVD
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from langdetect import detect
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
from scikeras.wrappers import KerasClassifier
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout,InputLayer,BatchNormalization
from tensorflow.keras.regularizers import l2
from keras.callbacks import EarlyStopping
from keras.optimizers import RMSprop
from lime.lime_text import LimeTextExplainer
#from imblearn.over_sampling import SMOTE




[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [95]:
# load the dataset
df = pd.read_csv('judge-1377884607_tweet_product_company.csv', encoding='latin1')
df.head()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


In [96]:
# check info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9093 entries, 0 to 9092
Data columns (total 3 columns):
 #   Column                                              Non-Null Count  Dtype 
---  ------                                              --------------  ----- 
 0   tweet_text                                          9092 non-null   object
 1   emotion_in_tweet_is_directed_at                     3291 non-null   object
 2   is_there_an_emotion_directed_at_a_brand_or_product  9093 non-null   object
dtypes: object(3)
memory usage: 213.2+ KB


In [97]:
# check for duplicates
df.duplicated().sum()

22

In [98]:
# check for missing values
df.isna().sum()

tweet_text                                               1
emotion_in_tweet_is_directed_at                       5802
is_there_an_emotion_directed_at_a_brand_or_product       0
dtype: int64

In [99]:
# drop duplicates
df.drop_duplicates()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion
...,...,...,...
9088,Ipad everywhere. #SXSW {link},iPad,Positive emotion
9089,"Wave, buzz... RT @mention We interrupt your re...",NaN,No emotion toward brand or product
9090,"Google's Zeiger, a physician never reported po...",NaN,No emotion toward brand or product
9091,Some Verizon iPhone customers complained their...,NaN,No emotion toward brand or product


In [100]:
# drop the emotion_in_tweet_is_directed_at as it contains much missing values
df.drop(columns='emotion_in_tweet_is_directed_at', inplace=True)

In [101]:
# rename columns
df = df.rename(columns={'tweet_text':'text','is_there_an_emotion_directed_at_a_brand_or_product': 'sentiment'})

In [102]:
# create another dataframe caaled df_sentiment
df_sentiment = df.copy()
df_sentiment

,text,sentiment
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Positive emotion
...,...,...
9088,Ipad everywhere. #SXSW {link},Positive emotion
9089,"Wave, buzz... RT @mention We interrupt your re...",No emotion toward brand or product
9090,"Google's Zeiger, a physician never reported po...",No emotion toward brand or product
9091,Some Verizon iPhone customers complained their...,No emotion toward brand or product


In [103]:
# check the value counts of sentiment
df_sentiment['sentiment'].value_counts()

sentiment
No emotion toward brand or product    5389
Positive emotion                      2978
Negative emotion                       570
I can't tell                           156
Name: count, dtype: int64

In [104]:
# remove whitespaces and convert the text column to string
df_sentiment['text'] = df_sentiment['text'].fillna('').astype(str)

# do feature engineering to find out the number of words, characters and sentences
df_sentiment['words'] = df_sentiment['text'].apply(len)
df_sentiment['chars'] = df_sentiment['text'].apply(lambda text: len(nltk.word_tokenize(text)))
df_sentiment['sent'] = df_sentiment['text'].apply(lambda text: len(nltk.sent_tokenize(text)))

df_sentiment

,text,sentiment,words,chars,sent
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,Negative emotion,127,32,5
1,@jessedee Know about @fludapp ? Awesome iPad/i...,Positive emotion,139,29,3
2,@swonderlin Can not wait for #iPad 2 also. The...,Positive emotion,79,20,2
3,@sxsw I hope this year's festival isn't as cra...,Negative emotion,82,21,2
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Positive emotion,131,29,1
...,...,...,...,...,...
9088,Ipad everywhere. #SXSW {link},Positive emotion,29,8,2
9089,"Wave, buzz... RT @mention We interrupt your re...",No emotion toward brand or product,125,26,1
9090,"Google's Zeiger, a physician never reported po...",No emotion toward brand or product,145,32,3
9091,Some Verizon iPhone customers complained their...,No emotion toward brand or product,140,26,2


In [105]:
# check for descriptive statistics
df_sentiment.describe()

,words,chars,sent
count,9093.000000,9093.000000,9093.000000
mean,104.950731,24.412295,1.880018
std,27.208419,6.505483,0.943527
min,0.000000,0.000000,0.000000
25%,86.000000,20.000000,1.000000
50%,109.000000,25.000000,2.000000
75%,126.000000,29.000000,2.000000
max,178.000000,49.000000,7.000000


In [106]:
# label encode the target
le = LabelEncoder()
df_sentiment['sentiment'] = le.fit_transform(df_sentiment['sentiment'])
df_sentiment['sentiment'] 


0       1
1       3
2       3
3       1
4       3
       ..
9088    3
9089    2
9090    2
9091    2
9092    2
Name: sentiment, Length: 9093, dtype: int32

In [132]:
# create an oop function to do data preprocessing

class SentimentPreprocessor:
    def __init__(self, text_column, target_column):
        self.text_column = text_column
        self.target_column = target_column
        self.vectorizer = TfidfVectorizer(ngram_range=(1,2))
        self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()

    def _is_english(self, text):
        try:
            return detect(text) == 'en'
        except:
            return False

    def _remove_emojis(self, text):
        emoji_pattern = re.compile("["  
                                   u"\U0001F600-\U0001F64F"
                                   u"\U0001F300-\U0001F5FF"
                                   u"\U0001F680-\U0001F6FF"
                                   u"\U0001F1E0-\U0001F1FF"
                                   "]+", flags=re.UNICODE)
        return emoji_pattern.sub(r'', text)

    def _clean_text(self, text):
        text = self._remove_emojis(text)
        text = re.sub(r'\d+', '', text)
        text = text.lower()
        text = text.translate(str.maketrans('', '', string.punctuation))
        return text

    def _tokenize(self, text):
        return word_tokenize(text)

    def _remove_stopwords(self, tokens):
        return [word for word in tokens if word.isalpha() and word not in self.stop_words]

    def _lemmatize_tokens(self, tokens):
        return [self.lemmatizer.lemmatize(token) for token in tokens]

    def preprocess(self, df_sentiment):
        print("1. Removing non-English rows")
        df_sentiment = df_sentiment[df_sentiment[self.text_column].astype(str).apply(self._is_english)].copy()

        print("2. Cleaning raw text")
        df_sentiment["clean_text"] = df_sentiment[self.text_column].astype(str).apply(self._clean_text)

        print("3. Tokenizing text")
        df_sentiment["tokenized_text"] = df_sentiment["clean_text"].apply(self._tokenize)

        print("4. Removing stopwords")
        df_sentiment["stopwords_removed"] = df_sentiment["tokenized_text"].apply(self._remove_stopwords)

        print("5. Lemmatizing tokens")
        df_sentiment["lemmatized_text"] = df_sentiment["stopwords_removed"].apply(self._lemmatize_tokens)

        print("6. Joining lemmatized tokens for final clean text")
        df_sentiment["final_clean_text"] = df_sentiment["lemmatized_text"].apply(lambda tokens: " ".join(tokens))

        print("7. Vectorizing 'final_clean_text' using TF-IDF")
        X_tfidf = self.vectorizer.fit_transform(df_sentiment["final_clean_text"])
        y = df_sentiment[self.target_column].values

        return df_sentiment, X_tfidf, y

    def split_and_process(self, df_sentiment, test_size=0.2):
        df_cleaned, X, y = self.preprocess(df_sentiment)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
        return df_cleaned, X_train, X_test, y_train, y_test




In [133]:
# apply the class on the data

processor = SentimentPreprocessor(text_column='text', target_column='sentiment')
df_cleaned, X_train, X_test, y_train, y_test = processor.split_and_process(df_sentiment)

# Print all relevant columns with intermediate steps
print(df_cleaned[['text', 'clean_text', 'tokenized_text', 'stopwords_removed', 'lemmatized_text', 'final_clean_text']].head())


1. Removing non-English rows
2. Cleaning raw text
3. Tokenizing text
4. Removing stopwords
5. Lemmatizing tokens
6. Joining lemmatized tokens for final clean text
7. Vectorizing 'final_clean_text' using TF-IDF
                                                text  \
0  .@wesley83 I have a 3G iPhone. After 3 hrs twe...   
1  @jessedee Know about @fludapp ? Awesome iPad/i...   
2  @swonderlin Can not wait for #iPad 2 also. The...   
3  @sxsw I hope this year's festival isn't as cra...   
4  @sxtxstate great stuff on Fri #SXSW: Marissa M...   

                                          clean_text  \
0  wesley i have a g iphone after  hrs tweeting a...   
1  jessedee know about fludapp  awesome ipadiphon...   
2  swonderlin can not wait for ipad  also they sh...   
3  sxsw i hope this years festival isnt as crashy...   
4  sxtxstate great stuff on fri sxsw marissa maye...   

                                      tokenized_text  \
0  [wesley, i, have, a, g, iphone, after, hrs, tw...   
1  [

In [152]:

# 1. Encode target
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# 2. Class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train_enc), y=y_train_enc)
class_weight_dict = dict(enumerate(class_weights))

# 3. Define Neural Network
def create_nn(input_dim):
    model = Sequential()
    model.add(InputLayer(input_shape=(input_dim,)))
    model.add(Dense(256, activation='relu', kernel_regularizer=l2(0.001)))
    model.add(Dropout(0.2))
    model.add(Dense(128, activation='relu', kernel_regularizer=l2(0.001)))
    model.add(Dropout(0.2))
    model.add(Dense(64, activation='relu', kernel_regularizer=l2(0.001)))
    model.add(Dense(len(le.classes_), activation='softmax'))
    
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# 4. Pipeline
svd_n_components = 450

pipelines = {
    "Decision Tree": Pipeline([
        ('svd', TruncatedSVD(n_components=svd_n_components)),
        ('model', DecisionTreeClassifier(class_weight='balanced'))
    ]),
    "Random Forest": Pipeline([
        ('svd', TruncatedSVD(n_components=svd_n_components)),
        ('model', RandomForestClassifier(class_weight='balanced'))
    ]),
    "XGBoost": Pipeline([
        ('svd', TruncatedSVD(n_components=svd_n_components)),
        ('model', XGBClassifier(scale_pos_weight=class_weights.max(), use_label_encoder=False, eval_metric='mlogloss'))
    ]),
    "Neural Net": Pipeline([
        ('svd', TruncatedSVD(n_components=svd_n_components)),
        ('scaler', StandardScaler()),
        ('model', KerasClassifier(
            model=create_nn,
            model__input_dim=svd_n_components,
            epochs=150,
            batch_size=32,
            verbose=0,
            validation_split=0.2,
            callbacks=[EarlyStopping(patience=5, restore_best_weights=True)],
            class_weight=class_weight_dict
        ))
    ])
}

# 5. Cross-validation
cv_scores = {}
print("Model Performance:")
for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train_enc, cv=3, scoring='accuracy')
    avg_score = scores.mean()
    cv_scores[name] = avg_score
    print(f"{name}: {avg_score:.2f} accuracy")

# 6. Final training
best_model_name = max(cv_scores, key=cv_scores.get)
print(f"Best Model: {best_model_name}")
best_model = pipelines[best_model_name].fit(X_train, y_train_enc)

# 7. Predict
sample_vector = X_train[0]
prediction = best_model.predict(sample_vector.reshape(1, -1))
predicted_label = le.inverse_transform(prediction)[0]

# 8. Output
print("Sample Prediction:")
print(f"Text: {df_cleaned.iloc[0]['final_clean_text'][:500]}")
print(f"Predicted: {predicted_label}")
print(f"Actual: {df_cleaned.iloc[0]['sentiment']}")


Model Performance:
Decision Tree: 0.51 accuracy
Random Forest: 0.63 accuracy


c:\Users\user\anaconda3\envs\ml_env\lib\site-packages\xgboost\training.py:183: UserWarning: [22:36:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "scale_pos_weight", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\user\anaconda3\envs\ml_env\lib\site-packages\xgboost\training.py:183: UserWarning: [22:37:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "scale_pos_weight", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\user\anaconda3\envs\ml_env\lib\site-packages\xgboost\training.py:183: UserWarning: [22:38:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "scale_pos_weight", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost: 0.65 accuracy


c:\Users\user\anaconda3\envs\ml_env\lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(
c:\Users\user\anaconda3\envs\ml_env\lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(
c:\Users\user\anaconda3\envs\ml_env\lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Neural Net: 0.38 accuracy
Best Model: XGBoost


c:\Users\user\anaconda3\envs\ml_env\lib\site-packages\xgboost\training.py:183: UserWarning: [22:39:24] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "scale_pos_weight", "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Sample Prediction:
Text: wesley g iphone hr tweeting riseaustin dead need upgrade plugin station sxsw
Predicted: 2
Actual: 1


In [154]:
# hyperparameter tune XGBoost

# Define XGBoost hyperparameter grid
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.01, 0.1, 0.3],
    'model__subsample': [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0]
}

# Build pipeline (scaling optional if input not sparse)
xgb_pipeline = Pipeline([
    ('model', XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42))
])

# Grid Search CV
grid_search = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=1,
    verbose=2
)

# Fit to training data
grid_search.fit(X_train, y_train_enc)

# Best model from grid search
best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

# Evaluate on test data
y_pred_train = best_model.predict(X_train)
y_pred_test = best_model.predict(X_test)
train_acc = accuracy_score(y_train_enc,y_pred_train)
test_acc = accuracy_score(y_test_enc, y_pred_test)

# Output
print("Grid Search with Pipeline Complete!")
print(f"Best Parameters: {best_params}")
print(f"Best Cross-Validation Accuracy: {best_score:.4f}")
print(f"Train Set Accuracy: {train_acc:.4f}")
print(f"Test Set Accuracy: {test_acc:.4f}")


Fitting 3 folds for each of 72 candidates, totalling 216 fits


c:\Users\user\anaconda3\envs\ml_env\lib\site-packages\xgboost\training.py:183: UserWarning: [09:45:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=100, model__subsample=0.8; total time=  10.1s


c:\Users\user\anaconda3\envs\ml_env\lib\site-packages\xgboost\training.py:183: UserWarning: [09:45:35] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=100, model__subsample=0.8; total time=  10.1s


...

[CV] END model__colsample_bytree=1.0, model__learning_rate=0.3, model__max_depth=7, model__n_estimators=200, model__subsample=1.0; total time=  47.9s


c:\Users\user\anaconda3\envs\ml_env\lib\site-packages\xgboost\training.py:183: UserWarning: [11:28:03] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END model__colsample_bytree=1.0, model__learning_rate=0.3, model__max_depth=7, model__n_estimators=200, model__subsample=1.0; total time=  47.2s


c:\Users\user\anaconda3\envs\ml_env\lib\site-packages\xgboost\training.py:183: UserWarning: [11:28:51] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Grid Search with Pipeline Complete!
Best Parameters: {'model__colsample_bytree': 0.8, 'model__learning_rate': 0.3, 'model__max_depth': 5, 'model__n_estimators': 200, 'model__subsample': 1.0}
Best Cross-Validation Accuracy: 0.6581
Train Set Accuracy: 0.8813
Test Set Accuracy: 0.6678
